# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata_json = dataset.metadata.to_json()
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record set @ids and basic info
from pprint import pprint

print("Record sets in the dataset:")
for rs in dataset.metadata.record_sets:
    print(f"@id: {rs.id}, name: {rs.name}, description: {rs.description}")

# Display field @ids for each record set
for rs in dataset.metadata.record_sets:
    print(f"\nFields for record set @id: {rs.id} ({rs.name}):")
    for field in rs.fields:
        print(f"  field @id: {field.id}, name: {field.name}, dataType: {field.data_type}, column: {field.column.id if hasattr(field, 'column') and field.column else 'N/A'}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets into dataframes dict, using @id as reference.
record_sets = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}
for record_set in record_sets:
    records = list(dataset.records(record_set=record_set))
    dataframes[record_set] = pd.DataFrame(records)

if record_sets:
    main_record_set_id = record_sets[0]  # Assume first is main, update by inspection!
    print(f"Columns in record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print('No record sets available in this dataset.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a numeric field by @id for filtering/analysis by inspecting the columns (modify as appropriate):
numeric_field = None

# Attempt to pick up a good numeric field if available
cols = dataframes[main_record_set_id].columns
potential_numeric_fields = [col for col in cols if 'abundance' in col.lower() or 'peptide' in col.lower() or 'mw' in col.lower() or 'number' in col.lower() or 'count' in col.lower() or 'coverage' in col.lower()]
if potential_numeric_fields:
    numeric_field = potential_numeric_fields[0]
else:
    # fallback: first numeric column
    numeric_cols = dataframes[main_record_set_id].select_dtypes(include='number').columns.tolist()
    if numeric_cols:
        numeric_field = numeric_cols[0]

print(f"Selected numeric field for EDA: {numeric_field}")

if numeric_field:
    threshold = dataframes[main_record_set_id][numeric_field].dropna().quantile(0.75)  # Use 75th percentile for filtering
    filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a possible categorical field
    group_field = None
    candidate_group_fields = [col for col in cols if col not in [numeric_field] and (('description' in col.lower()) or ('category' in col.lower()) or ('type' in col.lower()) or ('group' in col.lower()) or ('sample' in col.lower()))]
    if candidate_group_fields:
        group_field = candidate_group_fields[0]

    print(f"Grouping field chosen: {group_field}")

    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
else:
    print('No suitable numeric field found for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[main_record_set_id][numeric_field].dropna(), bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    if group_field and group_field in dataframes[main_record_set_id].columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field, data=dataframes[main_record_set_id])
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=30, ha='right')
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides mass spectrometry-based protein data with columns such as accession, abundance, and peptide counts, referenced by their unique `@id`s.
- We used `mlcroissant`'s schema navigation by `@id` for record sets and fields, and loaded tabular data into Pandas DataFrames for processing.
- We performed filtering and normalization of a quantitative variable, and visualized data distributions with Seaborn.

Further analyses can be performed by referencing new fields or relationships by their Croissant `@id`s as needed.